# Silver Curation

**Mission**: Build the `silver` schema in `/workspace/ICOCO.duckdb` by curating the bronze/raw tables.

**Principles**
- **Never modify bronze directly** — copy bronze → silver first, then process in silver (bronze stays as the recovery layer).
- **SQL-first** — use SQL wherever possible; use Python only when SQL is not effective (e.g. parsing Python-list literals stored as VARCHAR).
- **Reproducible** — every `CREATE` uses `IF NOT EXISTS` / `CREATE OR REPLACE` so the notebook can be re-run safely.

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100  # show all rows in query results
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL)
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

## 0. Create the silver schema

In [4]:
%%sql
CREATE SCHEMA IF NOT EXISTS silver;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


## 1. Add `coutries.csv` to bronze, then to silver

Source: `/workspace/data/coutries.csv` (columns: `Name`, `Code`).

In [5]:
%%sql
CREATE OR REPLACE TABLE bronze.countries AS
SELECT * FROM read_csv('/workspace/data/coutries.csv', header=true);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [6]:
%%sql
CREATE OR REPLACE TABLE silver.countries AS
SELECT * FROM bronze.countries;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [7]:
%%sql
SELECT * FROM silver.countries LIMIT 5;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Name,Code
Afghanistan,AF
Albania,AL
Algeria,DZ
American Samoa,AS
Andorra,AD


## 2. Add `food_category.csv` (latest & legacy) to bronze, then to silver

Sources:
- `/workspace/data/FoodUSDA_latest/food_category.csv`
- `/workspace/data/FoodUSDA_legacy/food_category.csv`

Both have the same columns: `id`, `code`, `description`.

In [8]:
%%sql
CREATE OR REPLACE TABLE bronze.FoodUSDA_latest_food_category AS
SELECT * FROM read_csv('/workspace/data/FoodUSDA_latest/food_category.csv', header=true);

CREATE OR REPLACE TABLE bronze.FoodUSDA_legacy_food_category AS
SELECT * FROM read_csv('/workspace/data/FoodUSDA_legacy/food_category.csv', header=true);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [9]:
%%sql
CREATE OR REPLACE TABLE silver.FoodUSDA_latest_food_category AS
SELECT * FROM bronze.FoodUSDA_latest_food_category;

CREATE OR REPLACE TABLE silver.FoodUSDA_legacy_food_category AS
SELECT * FROM bronze.FoodUSDA_legacy_food_category;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [10]:
%%sql
SELECT * FROM silver.FoodUSDA_latest_food_category LIMIT 5;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

id,code,description
1,0100,Dairy and Egg Products
2,0200,Spices and Herbs
3,0300,Baby Foods
4,0400,Fats and Oils
5,0500,Poultry Products


## 3. RecipeZenodo → `silver.RecipeZenodo`

**Goal**: keep only `Name`, `Category`, `Cuisine`, `Ingredients`, `Cooking Methods`.

**Trouble**: `Category`, `Cuisine`, `Ingredients`, `Cooking Methods` are stored as **VARCHAR** but actually contain Python list literals (e.g. `['Dinner']`), and `Ingredients` is a list of dicts (`[{'ingredient': 'pineapple', 'quantity': '1', ...}]`).

**Approach**:
1. Copy `bronze.RecipeZenodo` → `silver.RecipeZenodo` (full copy, bronze untouched).
2. Use Python `ast.literal_eval` to parse the list columns into real DuckDB `VARCHAR[]` lists (SQL string functions would be fragile here).
3. For `Ingredients`, keep only the ingredient **name** (drop quantity/unit/misc).

In [11]:
%%sql
CREATE OR REPLACE TABLE silver.RecipeZenodo AS
SELECT * FROM bronze.RecipeZenodo;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [12]:
import ast
from sql.connection import ConnectionManager

# IMPORTANT: DuckDB allows only ONE connection to the database at a time.
# Reuse the SAME connection that %sql already opened (do NOT duckdb.connect() again).
raw = ConnectionManager().current.connection_sqlalchemy.connection

# Read the full copy from silver (never touch bronze)
df = raw.execute('SELECT "Name", "Category", "Cuisine", "Ingredients", "Cooking Methods" FROM silver.RecipeZenodo').df()

def parse_list(x):
    """Parse a Python-list literal stored as VARCHAR into a real list."""
    try:
        v = ast.literal_eval(x)
        return v if isinstance(v, list) else []
    except Exception:
        return []

# Category / Cuisine / Cooking Methods are plain lists of strings
for col in ['Category', 'Cuisine', 'Cooking Methods']:
    df[col] = df[col].apply(parse_list)

# Ingredients is a list of dicts -> keep only the ingredient name
def parse_ingredients(x):
    try:
        v = ast.literal_eval(x)
        if isinstance(v, list):
            return [d['ingredient'] for d in v if isinstance(d, dict) and 'ingredient' in d]
    except Exception:
        pass
    return []

df['Ingredients'] = df['Ingredients'].apply(parse_ingredients)

# Write the curated table back to silver (CREATE OR REPLACE keeps it re-runnable)
# register() creates a temporary view; materialize it into the real table, then drop the view.
raw.register('df_zenodo', df)
raw.execute('CREATE OR REPLACE TABLE silver.RecipeZenodo AS SELECT * FROM df_zenodo')
raw.execute('DROP VIEW IF EXISTS df_zenodo')

print('silver.RecipeZenodo rows:', len(df))
print(df.head(2).to_string())

silver.RecipeZenodo rows: 50511
                      Name             Category         Cuisine                                                                                                                                                             Ingredients Cooking Methods
0  Pineapple Glaze for Ham             [Dinner]      [American]                                                                                                                           [pineapple, maraschino cherries, brown sugar]     [microwave]
1        Awesome Egg Rolls  [Dinner, Appetizer]  [World, Asian]  [vegetable oil, egg, cabbage, bean sprouts, carrot, stalk, onion, shrimp, soy sauce, garlic powder, pepper to taste, cornstarch, oil for frying, water, roll wrappers]           [fry]


In [13]:
%%sql
SELECT "Name", "Category", "Cuisine", "Ingredients", "Cooking Methods"
FROM silver.RecipeZenodo
LIMIT 3;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Name,Category,Cuisine,Ingredients,Cooking Methods
Pineapple Glaze for Ham,['Dinner'],['American'],"['pineapple', 'maraschino cherries', 'brown sugar']",['microwave']
Awesome Egg Rolls,"['Dinner', 'Appetizer']","['World', 'Asian']","['vegetable oil', 'egg', 'cabbage', 'bean sprouts', 'carrot', 'stalk', 'onion', 'shrimp', 'soy sauce', 'garlic powder', 'pepper to taste', 'cornstarch', 'oil for frying', 'water', 'roll wrappers']",['fry']
Spanish Tortilla,['Breakfast'],['Spanish'],"['olive oil', 'potatoes', 'bacon', 'ham', 'onion', 'bell pepper', 'eggs', 'spanish seasoning']",['fry']


In [14]:
%%sql
DESCRIBE silver.RecipeZenodo;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

column_name,column_type,null,key,default,extra


## 4. FooDB.flavor → `silver.FooDB_flavor`

Keep only: `id`, `name`, `flavor_group`, `category`.

In [15]:
%%sql
CREATE OR REPLACE TABLE silver.FooDB_flavor AS
SELECT id, name, flavor_group, category
FROM FooDB.flavor;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [16]:
%%sql
SELECT * FROM silver.FooDB_flavor LIMIT 5;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

id,name,flavor_group,category
1,celery,vegetable,odor
2,corn,vegetable,odor
3,cucumber,vegetable,odor
4,horseradish,vegetable,odor
5,vegetable,vegetable,odor


## 5. FoodUSDA (latest & legacy) → silver

For **both** `FoodUSDA_latest` and `FoodUSDA_legacy`:

| Source table | Keep columns | Silver table |
|---|---|---|
| `food` | `fdc_id`, `data_type`, `description`, `food_category_id` | `silver.FoodUSDA_<version>_food` |
| `nutrient` | `id`, `name` | `silver.FoodUSDA_<version>_nutrient` |
| `food_nutrient` | `id`, `fdc_id`, `nutrient_id` | `silver.FoodUSDA_<version>_food_nutrient` |

In [17]:
%%sql
-- FoodUSDA_latest
CREATE OR REPLACE TABLE silver.FoodUSDA_latest_food AS
SELECT fdc_id, data_type, description, food_category_id FROM FoodUSDA_latest.food;

CREATE OR REPLACE TABLE silver.FoodUSDA_latest_nutrient AS
SELECT id, name FROM FoodUSDA_latest.nutrient;

CREATE OR REPLACE TABLE silver.FoodUSDA_latest_food_nutrient AS
SELECT id, fdc_id, nutrient_id FROM FoodUSDA_latest.food_nutrient;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [18]:
%%sql
-- FoodUSDA_legacy
CREATE OR REPLACE TABLE silver.FoodUSDA_legacy_food AS
SELECT fdc_id, data_type, description, food_category_id FROM FoodUSDA_legacy.food;

CREATE OR REPLACE TABLE silver.FoodUSDA_legacy_nutrient AS
SELECT id, name FROM FoodUSDA_legacy.nutrient;

CREATE OR REPLACE TABLE silver.FoodUSDA_legacy_food_nutrient AS
SELECT id, fdc_id, nutrient_id FROM FoodUSDA_legacy.food_nutrient;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [19]:
%%sql
SELECT * FROM silver.FoodUSDA_latest_food LIMIT 3;
SELECT * FROM silver.FoodUSDA_latest_nutrient LIMIT 3;
SELECT * FROM silver.FoodUSDA_latest_food_nutrient LIMIT 3;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

id,fdc_id,nutrient_id
2201847,319877,1051
2201845,319877,1002
2201846,319877,1004


## 6. Verification — all silver tables

In [20]:
%%sql
SELECT table_name, row_count FROM (
    SELECT 'silver.countries' AS table_name, COUNT(*) AS row_count FROM silver.countries
    UNION ALL SELECT 'silver.FoodUSDA_latest_food_category', COUNT(*) FROM silver.FoodUSDA_latest_food_category
    UNION ALL SELECT 'silver.FoodUSDA_legacy_food_category', COUNT(*) FROM silver.FoodUSDA_legacy_food_category
    UNION ALL SELECT 'silver.RecipeZenodo', COUNT(*) FROM silver.RecipeZenodo
    UNION ALL SELECT 'silver.FooDB_flavor', COUNT(*) FROM silver.FooDB_flavor
    UNION ALL SELECT 'silver.FoodUSDA_latest_food', COUNT(*) FROM silver.FoodUSDA_latest_food
    UNION ALL SELECT 'silver.FoodUSDA_latest_nutrient', COUNT(*) FROM silver.FoodUSDA_latest_nutrient
    UNION ALL SELECT 'silver.FoodUSDA_latest_food_nutrient', COUNT(*) FROM silver.FoodUSDA_latest_food_nutrient
    UNION ALL SELECT 'silver.FoodUSDA_legacy_food', COUNT(*) FROM silver.FoodUSDA_legacy_food
    UNION ALL SELECT 'silver.FoodUSDA_legacy_nutrient', COUNT(*) FROM silver.FoodUSDA_legacy_nutrient
    UNION ALL SELECT 'silver.FoodUSDA_legacy_food_nutrient', COUNT(*) FROM silver.FoodUSDA_legacy_food_nutrient
) ORDER BY table_name;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

table_name,row_count
silver.FooDB_flavor,883
silver.FoodUSDA_latest_food,87990
silver.FoodUSDA_latest_food_category,28
silver.FoodUSDA_latest_food_nutrient,170469
silver.FoodUSDA_latest_nutrient,477
silver.FoodUSDA_legacy_food,7793
silver.FoodUSDA_legacy_food_category,28
silver.FoodUSDA_legacy_food_nutrient,644125
silver.FoodUSDA_legacy_nutrient,474
silver.RecipeZenodo,50511
